# Data load & Sanity Check

The model (`donderformer.models.taikonation`) and train/validation loops (`donderformer.training`) live in `src/`. This notebook trains and saves a run. Open `06_evaluation.ipynb` in a fresh kernel to evaluate that saved run without retraining. The original full-data PACE run is in `notebooks/reference/baseline_cnn_lstm_full_pace.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from donderformer.data.feature_extractor import FeatureExtractor
from donderformer.data.taiko_dataset import Window
from donderformer.paths import DataPaths, outputs_dir
from donderformer.runs import new_run_dir

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for a quick local run.
paths = DataPaths()
SPLIT = "oni_v2"                              # folder under splits/, made by 01_make_split
extractor = FeatureExtractor(frame_ms=23)     # cache made by 02_extract_features

# Window, in 23 ms frames, around the first predicted frame i. Defaults are the paper's:
# 16 audio frames ending at i, the 15 previous notes, predict i .. i+3.
AUDIO_FRAMES = 16    # audio frames per window
HISTORY = 15         # previous note frames fed back in (<= AUDIO_FRAMES - 1 - LOOK_FORWARD)
LOOK_FORWARD = 0     # audio frames after i; 3 lets the audio cover all 4 predicted frames
WINDOW = Window(audio_frames=AUDIO_FRAMES, history=HISTORY, look_forward=LOOK_FORWARD)
print("Window:", WINDOW.describe())

DROPOUT = 0.2        # drop probability; the released code's keep_prob=0.8

MODEL_NAME = "taikonation"
ALIAS = WINDOW.name                           # Something descriptive, such as Taikonation_baseline, etc. or something about the windown length

# Every run gets its own folder: outputs/runs/<model>/<datetime>[-<alias>]-<split>-<extractor>/
RUN_DIR = new_run_dir(MODEL_NAME, SPLIT, extractor.id, alias=ALIAS)
print("Run folder:", RUN_DIR)
DATA_INFO = {"split": SPLIT, "features": extractor.id, "window": WINDOW.to_dict()}

# Training. Defaults are the full-data PACE run; TaikoNation's paper fine-tunes at batch size 1.
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-5
FINETUNE_BATCH_SIZE = 128
FINETUNE_EPOCHS = 2
FINETUNE_LR = 5e-6
HPARAMS = dict(batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, finetune_batch_size=FINETUNE_BATCH_SIZE,
               finetune_epochs=FINETUNE_EPOCHS, finetune_lr=FINETUNE_LR)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# local setup
# device = torch.device(
#     "mps" if torch.backends.mps.is_available()
#     else "cuda" if torch.cuda.is_available()
#     else "cpu"
# )

# print("Device:", device)

# PACE - cuda expected
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)


if device.type == "cuda":
    p = torch.cuda.get_device_properties(device)
    print(f"{torch.cuda.get_device_name(device)} | {p.total_memory / 1e9:.0f} GB | {p.multi_processor_count} SMs | compute {p.major}.{p.minor}")
    print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(), "| GPUs visible:", torch.cuda.device_count())
    print(f"in use by this kernel: {torch.cuda.memory_allocated() / 1e9:.2f} GB (peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB)")

### Model Pipeline

**One input window**  
↓  
**`AUDIO_FRAMES` audio frames (A × 80) + `HISTORY` previous note frames (H × 7)**  
↓  
**CNN extracts features from the audio**  
↓  
**Audio features are combined with previous note information**  
↓  
**LSTM learns temporal/rhythmic patterns**  
↓  
**Predict 4 timesteps (4 × 7 classes)**

In [ ]:
# Defined in src/donderformer/models/taikonation.py
from donderformer.models.taikonation import TaikoNationBaseline

# sanity check

In [ ]:
model = TaikoNationBaseline(WINDOW, dropout=DROPOUT).to(device)
from donderformer.data.taiko_dataset import TaikoDataset
sample_ds = TaikoDataset(paths, SPLIT, extractor, "train", WINDOW)
X, _ = sample_ds.chart_windows(sample_ds.chart_ids[0])
sample = torch.from_numpy(X[:8]).to(device)
with torch.no_grad():
    output = model(sample)
print(WINDOW.describe())
print("Input:", sample.shape, "Output:", output.shape)
assert sample.shape[1] == model.n_inputs
assert output.shape == (sample.shape[0], WINDOW.targets, 7)

# data loader

Songs are loaded one at a time (see `donderformer.data.taiko_dataset`) because the full set is ~2k charts.

In [ ]:
from torch.utils.data import DataLoader
from donderformer.data.taiko_dataset import SongBatchSampler, TaikoDataset

train_dataset = TaikoDataset(paths, SPLIT, extractor, "train", WINDOW)
val_dataset = TaikoDataset(paths, SPLIT, extractor, "validation", WINDOW)
train_loader = DataLoader(train_dataset,
    batch_sampler=SongBatchSampler(train_dataset, BATCH_SIZE, shuffle=True), num_workers=0)
val_loader = DataLoader(val_dataset,
    batch_sampler=SongBatchSampler(val_dataset, BATCH_SIZE), num_workers=0)
for name, ds in (("train", train_dataset), ("validation", val_dataset)):
    print(f"{name}: {len(ds.chart_ids)} charts, {len(ds)} windows")
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

# loss + optimizer

In [ ]:
# measures how wrong the model's predictions are compared withthe correct note classes
criterion = nn.CrossEntropyLoss()

# changes the model's weights to reduce the loss; Adam is a popular optimizer that works well in many situations
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

# training & validation functions

In [ ]:
# predict -> calculate loss -> backpropagate -> update weights
# Defined in src/donderformer/training.py
from donderformer.training import evaluate, train_epoch

# Train (stage 1)

In [ ]:
import time
start_time = time.time()

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Elapsed time: {(time.time() - start_time) / 60:.1f} min"
    )

# Second stage - fine tuning

TaikoNation fine-tunes with batch size 1 and learning rate 5e-6. The full-data PACE run used `FINETUNE_BATCH_SIZE = 128` for 2 epochs, since batch 1 over ~7M windows takes days; set it to 1 in the config cell to replicate the paper.

In [ ]:
train_loader_finetune = DataLoader(
    train_dataset,
    batch_sampler=SongBatchSampler(train_dataset, FINETUNE_BATCH_SIZE, shuffle=True),
    num_workers=0
)
optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR)

finetune_train_losses = []
finetune_val_losses = []

for epoch in range(FINETUNE_EPOCHS):
    train_loss = train_epoch(model, train_loader_finetune, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    finetune_train_losses.append(train_loss)
    finetune_val_losses.append(val_loss)
    print(
        f"Fine-tune Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Elapsed time: {(time.time() - start_time) / 60:.1f} min"
    )

# save model

In [ ]:
from donderformer.runs import save_run

checkpoint = save_run(
    RUN_DIR,
    model,
    config={"model": MODEL_NAME, "alias": ALIAS, "data": DATA_INFO, "dropout": DROPOUT,
            "hparams": HPARAMS, "device": str(device)},
    losses={
        "train": train_losses,
        "val": val_losses,
        "finetune_train": finetune_train_losses,
        "finetune_val": finetune_val_losses,
    },
)

print("Saved", checkpoint)

The run now contains `config.json` and `model.pt` (weights, losses and embedded config). Copy the printed run folder into `RUN_DIR` in `06_evaluation.ipynb`. Keep the matching split manifest and feature cache, including train normalization and any failure list. Evaluation writes results under this same run; notebook 07 exports those generated charts.